Imports required modules and formats output

In [ ]:
import sys
import os

sys.path.append('../SlitMaker')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import matplotlib as mpl
import matplotlib.image as mpimg
from matplotlib.collections import LineCollection
%matplotlib inline
from scipy.optimize import curve_fit
from scipy.optimize import minimize_scalar
import pandas as pd
from astropy.io import fits
from astropy.wcs import WCS
from astropy.modeling.models import BlackBody
from astropy.constants import c
from astropy import units as u
from astropy.units import photometric
from synphot import SourceSpectrum, Observation, SpectralElement
from synphot.units import VEGAMAG
import slitoverlaymaker
import overlayfunction
import math

In [ ]:
mpl.rcParams['font.family']='serif'
mpl.rcParams['font.size']=14

Creates templates for the CMDs and RA-DEC plots.

In [ ]:
def InitializeCMDPlot():
    plt.clf()
    f= plt.figure(figsize=(8,6))
    plt.xlim(-1.5, 3.0)
    plt.ylim(29, 22)
    plt.xlabel('F606W - F814W (mag)')
    plt.ylabel('F814W (mag)')

In [ ]:
def InitializeRADECPlot():
    plt.clf()
    f= plt.figure(figsize=(8,8))
    #plt.xlim(0.15, -0.08)
    #plt.ylim(-0.04, 0.05)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.xlabel('Delta RA (deg)')
    plt.ylabel('Delta DEC (deg)')

Sets the galaxy center and the scaling factor

In [ ]:
centerRA = 151.2804167
centerDEC = 66.5550000
factor = np.cos(math.radians(centerDEC))

Extracts data from the .fits file and the mask output file to match spectroscopic targets.

In [ ]:
kdg63 = fits.open("hlsp_angst_hst_acs-wfc_9884-ddo71_f606w-f814w_v1_gst.fits")
starData = kdg63[1].data
f606w = []
f814w = []
allRA = []
allDEC = []
for i in starData:
    f606w.append(i[5])
    f814w.append(i[13])
    allRA.append(i[2])
    allDEC.append(i[3])
f606w = np.array(f606w)
f814w = np.array(f814w)
allRA = np.array(allRA)
allDEC = np.array(allDEC)
color = f606w - f814w

In [ ]:
selectedData = pd.read_csv('KDG63A.txt', sep=" ")

selectedRA = selectedData["RA"]
raDegrees = []
for entry in selectedRA:
    h,m,s = entry.split(":")
    coordinate = 15*float(h) + 15.0*float(m)/60.0 + 15.0*float(s)/3600.0
    raDegrees.append(coordinate)
raDegrees = np.array(raDegrees)

selectedDEC = selectedData["DEC"]
decDegrees= []
for entry in selectedDEC:
    h,m,s = entry.split(":")
    _,h = h.split("+")
    coordinate = float(h) + float(m)/60.0 + float(s)/3600.0
    decDegrees.append(coordinate)
decDegrees = np.array(decDegrees)

RAplotting = []
index = 0
for i in range(len(allRA)):
    index += 1
    for j in range(len(raDegrees)):
        if (abs(allRA[i] - raDegrees[j]) < 0.000021) and (abs(allDEC[i] - decDegrees[j]) < 0.000021):
            RAplotting.append(True)
    if len(RAplotting) < index :
        RAplotting.append(False)
count = 0
for i in range(len(RAplotting)):
    if RAplotting[i]:
        count += 1
print(count)

Plots a CMD for KDG63 with the spectroscopic targets shown.

In [ ]:
f1 = InitializeCMDPlot()
plt.scatter(color, f814w, c = 'gray', s = 3, edgecolors = 'none')
plt.scatter(color[RAplotting], f814w[RAplotting], c = "red", s = 10, edgecolors = "black")
plt.show()

Defines boundary lines for the HST image.

In [ ]:
deltaRA = (allRA - centerRA)*factor
deltaDEC = allDEC - centerDEC

l1 = [((-0.002 + 0.086)*factor, 0.0431 - 0.0221), ((-0.109 + 0.086)*factor, 0.0053 - 0.0221)]
l2 = [((-0.054 + 0.086)*factor, 0.0633 - 0.0221), ((-0.161 + 0.086)*factor, 0.0255 - 0.0221)]
l3 = [((-0.002 + 0.086)*factor, 0.0431 - 0.0221), ((-0.054 + 0.086)*factor, 0.0633 - 0.0221)]
l4 = [((-0.109 + 0.086)*factor, 0.0053 - 0.0221), ((-0.161 + 0.086)*factor, 0.0255 - 0.0221)]
l5 = [((0.049 + 0.086)*factor, 0.023 - 0.0221), ((-0.056 + 0.086)*factor, -0.0139 - 0.0221)]
l6 = [((0.049 + 0.086)*factor, 0.023 - 0.0221), ((-0.002 + 0.086)*factor, 0.0431 - 0.0221)]
l7 = [((-0.109 + 0.086)*factor, 0.0053 - 0.0221), ((-0.056 + 0.086)*factor, -0.0139 - 0.0221)]
lc = LineCollection([l1, l2, l3, l4, l5, l6, l7], color='k',lw=2)

def line(coord):
    x1 = coord[0][0]
    y1 = coord[0][1]
    x2 = coord[1][0]
    y2 = coord[1][1]
    slope = (y2 - y1)/(x2 - x1)
    intercept = y1 - slope*x1
    return slope, intercept

def f1(x):
    slope, intercept = line(l1)
    return x*slope + intercept
def f2(x):
    slope, intercept = line(l2)
    return x*slope + intercept
def f3(x):
    slope, intercept = line(l3)
    return x*slope + intercept
def f4(x):
    slope, intercept = line(l4)
    return x*slope + intercept
def f5(x):
    slope, intercept = line(l5)
    return x*slope + intercept
def f6(x):
    slope, intercept = line(l6)
    return x*slope + intercept
def f7(x):
    slope, intercept = line(l7)
    return x*slope + intercept
f1 = np.vectorize(f1)
f2 = np.vectorize(f2)
f3 = np.vectorize(f3)
f4 = np.vectorize(f4)
f5 = np.vectorize(f5)
f6 = np.vectorize(f6)
f7 = np.vectorize(f7)

def isInBox(x, y):
    flag = False
    if (y >= f1(x)) & (y <= f2(x)) & (y <= f3(x)) & (y >= f4(x)):
        flag = True
    if (y <= f1(x)) & (y >= f5(x)) & (y <= f6(x)) & (y >= f7(x)):
        flag = True
    return flag
isInBox = np.vectorize(isInBox)

inBox = isInBox(deltaRA, deltaDEC)
count = 0
for i in inBox:
    if i:
        count += 1
count/len(deltaRA)

Plots an RA-DEC spatial diagram of KDG63 with boundary lines and the spectroscopic targets.

In [ ]:
f2 = InitializeRADECPlot()
plt.scatter(deltaRA, deltaDEC, c = 'gray', s = 3, edgecolors = 'none')
plt.scatter(deltaRA[RAplotting], deltaDEC[RAplotting], c = "red", s = 10, edgecolors = "black")
RAcenter = (151.2804167 - centerRA)*factor
DECcenter = 66.5550000 - centerDEC
plt.scatter(RAcenter, DECcenter, c = "green", s=10)
plt.gca().add_collection(lc)
plt.gca().invert_xaxis()
plt.show()

Generates a random array of points and calculates the density using the annulus method.

In [ ]:
randRAArray = np.random.rand(1000000)
randDECArray = np.random.rand(1000000)
randRAArray = randRAArray*0.1 - 0.04
randDECArray = randDECArray*0.09 - 0.04

def distance(ra, dec):
    return np.sqrt(ra**2 + dec**2)
vfunc = np.vectorize(distance)

randomInBox = isInBox(randRAArray, randDECArray)

In [ ]:
def calculateDensity(innerRadius, outerRadius):
    rectArea = 0.1 * 0.09
    starsInsideRing = np.logical_and(vfunc(deltaRA, deltaDEC) > innerRadius, vfunc(deltaRA, deltaDEC) < outerRadius)
    pointsInsideRing = np.logical_and(vfunc(randRAArray, randDECArray) < outerRadius,
                                      vfunc(randRAArray, randDECArray) > innerRadius)

    pointsInsideCutRing = pointsInsideRing & randomInBox

    starCount = 0
    for i in starsInsideRing:
        if i:
            starCount += 1

    pointCount = 0
    for i in pointsInsideCutRing:
        if i:
            pointCount += 1

    if pointCount == 0:
        pointCount = 1
    return starCount/(rectArea*pointCount/1000000)

densities = []
radii = []
for medianRadius in np.arange(0.001, 0.05, 0.002):
    densities.append(calculateDensity(medianRadius - 0.001, medianRadius + 0.001))
    radii.append(medianRadius)

Plots the star density of KDG63 against the median radii of the annuli.

In [ ]:
plt.plot(radii, densities, marker='o', linestyle='-', color='r', label='Data')

plt.xlabel('Radius')
plt.ylabel('Density')
plt.title('Plot of Array Against Radius')
plt.yscale('log')
plt.xscale('log')
plt.show()

Creates an ellipse to demarcate the edges of the galaxy to create CMDs of the stars inside and outside the galaxy separately and shows the ellipse on top of an RA-DEC spatial diagram.

In [ ]:
centerRADistance = np.abs(deltaRA)
centerDECDistance = np.abs(deltaDEC)

def radius(theta, ra, dec):
    a = ra
    b = dec
    return a*b/np.sqrt((b*np.cos(theta))**2 + (a*np.sin(theta))**2)

thetas = np.arctan(deltaDEC/deltaRA)
distances = vfunc(deltaRA, deltaDEC)

starsInside = (distances < radius(thetas, np.percentile(centerRADistance, 90), np.percentile(centerDECDistance, 90)))
starsOutside = np.logical_not(starsInside)

f3 = InitializeCMDPlot()
plt.scatter(color[starsInside], f814w[starsInside], c = 'gray', s = 3, edgecolors = 'none')
plt.scatter(color[RAplotting & starsInside], f814w[RAplotting & starsInside], c = "red", s = 10, edgecolors = "black", label="Spectroscopic Targets")
plt.legend()
plt.show()

f4 = InitializeCMDPlot()
plt.scatter(color[starsOutside], f814w[starsOutside], c = 'red', s = 3, edgecolors = 'none')
plt.scatter(color[RAplotting & starsOutside], f814w[RAplotting & starsOutside], c = "blue", s = 10, edgecolors = "black", label="Spectroscopic Targets")
plt.legend()
plt.show()

In [ ]:
f5 = InitializeRADECPlot()
plt.scatter(deltaRA, deltaDEC, c = 'gray', s = 1, edgecolors = 'none')
plt.scatter(deltaRA[RAplotting], deltaDEC[RAplotting], c = "red", s = 10, edgecolors = "black", label="Spectroscopic Targets")
#plt.scatter(151.2923333 - centerRA, 66.5555833 - centerDEC, c = "green", s = 10, edgecolors = "black", label="Spectroscopic Targets")
plt.scatter(0, 0, c = "blue", s=10)
circle = Ellipse((0, 0), 2*np.percentile(centerRADistance, 90), 2*np.percentile(centerDECDistance, 90), fill=False, label="Characteristic Radius")
plt.gca().add_patch(circle)
#plt.gca().add_collection(lc)
#plt.legend()
plt.gca().invert_xaxis()
plt.show()

Creates a slit around the target stars on the RA-DEC map.

In [ ]:
def makePlot(fileName):
    hdu = fits.open(fileName)
    data = hdu[2].header
    
    rh,rm,rs = data['RA_OBJ'].split(":")
    targetRA = 15*float(rh) + 15.0*float(rm)/60.0 + 15.0*float(rs)/3600.0
    targetRA = (targetRA - centerRA)*factor
    
    dh,dm,ds = data['DEC_OBJ'].split(":")
    _,dh = dh.split("+")
    targetDEC = float(dh) + float(dm)/60.0 + float(ds)/3600.0
    targetDEC = targetDEC - centerDEC
    
    pa = -1*data['SLITPA']
    
    length = (data['SLITX1'] - data['SLITX0'])*0.1185/3600
    length_pos = ((hdu[3].header['OBJPOS'] + hdu[4].header['OBJPOS'])/2)*0.1185/3600
    print(hdu)

    x,y = boxPoints(targetRA, targetDEC, length_pos, length, pa)

    p1 = [(x[0],y[0]), (x[1],y[1])]
    p2 = [(x[2],y[2]), (x[3],y[3])]
    p3 = [(x[0],y[0]), (x[2],y[2])]
    p4 = [(x[1],y[1]), (x[3],y[3])]
    lc2 = LineCollection([p1, p2, p3, p4], color='orange',lw=2)

    f6 = InitializeRADECPlot()
    plt.scatter(deltaRA, deltaDEC, c = 'gray', s = 3, edgecolors = 'none')
    plt.scatter(targetRA, targetDEC, c = "red", s = 10, edgecolors = "black")
    plt.gca().add_collection(lc2)
    plt.gca().invert_xaxis()
    plt.xlim(targetRA + 0.005, targetRA - 0.005)
    plt.ylim(targetDEC - 0.005, targetDEC + 0.005)
    plt.show()

def rotate_point(x, y, cx, cy, angle_rad):
        x_new = cx + (x - cx) * math.cos(angle_rad) - (y - cy) * math.sin(angle_rad)
        y_new = cy + (x - cx) * math.sin(angle_rad) + (y - cy) * math.cos(angle_rad)
        return x_new, y_new

def boxPoints(targetRA, targetDEC, length_pos, length, pa):
    x0 = targetRA - 0.4/3600
    x1 = targetRA + 0.4/3600
    y0 = targetDEC - length_pos
    y1 = targetDEC + (length - length_pos)

    rotated_corners = np.array([
        rotate_point(x0, y0, targetRA, targetDEC, pa),  # x0y0
        rotate_point(x0, y1, targetRA, targetDEC, pa),  # x0y1
        rotate_point(x1, y0, targetRA, targetDEC, pa),  # x1y0
        rotate_point(x1, y1, targetRA, targetDEC, pa)   # x1y1
    ])

    x_coords = rotated_corners[:, 0]
    y_coords = rotated_corners[:, 1]
    return x_coords,y_coords

In [ ]:
name = './spec1dKDG63A/spec1d.KDG63A.017.pr1yng0137.fits'
makePlot(name)

Calculates the flux at points based on the magnitudes of the stars and a gaussian distribution within the region surrounding the slit. 

In [ ]:
def mag_to_flux(filter_name, mag):
    hst_calibration = {
        'F606W': {
            'PHOTFLAM': 7.9016e-20 * u.erg / (u.s * u.cm**2 * u.AA),
            'VEGAMAG_ZP': 26.399
        },
        'F814W': {
            'PHOTFLAM': 7.1340e-20 * u.erg / (u.s * u.cm**2 * u.AA),
            'VEGAMAG_ZP': 25.500
        }
    }
    
    if filter_name not in hst_calibration:
        raise ValueError("Filter name must be 'F606W' or 'F814W'")

    cal = hst_calibration[filter_name]
    photflam = cal['PHOTFLAM']
    zp = cal['VEGAMAG_ZP']

    flux = photflam * 10**(-0.4 * (mag - zp))
    return flux

def modelFluxRatio(T, wl1, wl2):
    bb = BlackBody(temperature=T * u.K, scale=1.0 * u.erg/(u.s * u.cm**2 * u.AA * u.sr))
    f1 = bb(wl1 * u.AA)
    f2 = bb(wl2 * u.AA)
    return (f1 / f2).value

def temperatureFit(obs_ratio, wl1, wl2):
    def loss(T):
        model_ratio = modelFluxRatio(T, wl1, wl2)
        return (model_ratio - obs_ratio)**2

    res = minimize_scalar(loss, bounds=(2000, 10000), method='bounded')
    return res.x

def checkModelAccuracy(T, scaleFactor, fluxf606w, fluxf814w):
    filterdataf606w = np.loadtxt("HST_ACS_HRC.F606W.dat")
    wavelengthsf606w = filterdataf606w[:, 0] * u.AA
    throughputf606w = filterdataf606w[:, 1]
    
    filterdataf814w = np.loadtxt("HST_ACS_HRC.F814W.dat")
    wavelengthsf814w = filterdataf606w[:, 0] * u.AA
    throughputf814w = filterdataf606w[:, 1]

    bb = BlackBody(temperature=T * u.K, scale=1.0 * u.erg/(u.s * u.cm**2 * u.AA * u.sr))
    bb_flux = bb(wavelengths) * (1.0 * u.sr)

    numeratorf606w = (bb_flux * throughputf606w)
    integrated_fluxf606w = np.trapz(numeratorf606w, wavelengthsf606w)
    normalizationf606w = np.trapz(throughputf606w, wavelengthsf606w)

    numeratorf814w = (bb_flux * throughputf814w)
    integrated_fluxf814w = np.trapz(numeratorf814w, wavelengthsf814w)
    normalizationf814w = np.trapz(throughputf814w, wavelengthsf814w)

    mean_fluxf606w = (integrated_fluxf606w / normalizationf606w) * scaleFactor
    print(type(mean_fluxf606w))
    mean_fluxf814w = (integrated_fluxf814w / normalizationf814w) * scaleFactor
    print(
        (mean_fluxf606w.value - fluxf606w.value) / mean_fluxf606w.value,
        (mean_fluxf814w.value - fluxf814w.value) / mean_fluxf814w.value
         )

def wavelengthFlux(mag_f606w, mag_f814w, wavelength):
    f606w_flux = mag_to_flux('F606W', mag_f606w)
    f814w_flux = mag_to_flux('F814W', mag_f814w)
    print(f606w_flux, f814w_flux)
    ratio = (f606w_flux/f814w_flux)

    T = temperatureFit(ratio, 5809.26, 7973.39)

    bb = BlackBody(temperature=T * u.K, scale=1.0 * u.erg/(u.s * u.cm**2 * u.AA * u.sr))
    totalFlux = bb(wavelength * u.AA)
    scalingFactor = f606w_flux / bb(5809.26 * u.AA)
    checkModelAccuracy(T, scalingFactor, f606w_flux, f814w_flux)

    return totalFlux.value*scalingFactor

def starGaussian(ra, dec, ra0, dec0, sigma, flux):
    dra = (ra - ra0) * 3600  # arcsec
    ddec = (dec - dec0) * 3600
    r2 = dra**2 + ddec**2
    spatial = (1/(2 * np.pi * sigma**2)) * np.exp(-r2/(2 * sigma**2))  # arcsec⁻²
    return (flux * spatial)

In [ ]:
sky_sigma = 1.2
flux = wavelengthFlux(22.819, 23.541, 4000)
print(flux)
#print(starGaussian(151.25917053222656, 66.5535659790039)

In [ ]:
hdu = fits.open('./spec1dKDG63A/spec1d.KDG63A.017.pr1yng0137.fits')
data = hdu[1].data
print(hdu[1].header.tostring(sep='\n'))
#print(data['FWHM'])

directory = "./spec1dKDG63A"

for filename in os.listdir(directory):
    filepath = os.path.join(directory, filename)  # Full path to the file
    if os.path.isfile(filepath):  # Check if it's a file (not a folder)
        hdu = fits.open(filepath)
        header1 = hdu[1].header
        data1 = hdu[1].data
        data2 = hdu[2].data
        data3 = hdu[3].data
        data4 = hdu[4].data
        header2 = hdu[2].header
        header3 = hdu[3].header
        header4 = hdu[4].header
        #print(data1['FWHM'], data2['FWHM'], data3['FWHM'], data4['FWHM'])

In [ ]:
plt.clf()
f= plt.figure(figsize=(8,8))
plt.plot(np.linspace(0, header1['SLITX1']-header1['SLITX0'], data1['SPEC'].size), data1['SPEC'][0],  c = 'gray')
#plt.xlim(targetRA + 0.005, targetRA - 0.005)
#plt.ylim(targetDEC - 0.005, targetDEC + 0.005)
plt.show()